Question 7 

Image Classification Pipeline 

Create a reusable Python pipeline that loads an image dataset, resizes images, normalizes pixel values, splits the data into train/validation sets, and stores metadata in a CSV. 

Expected Deliverables: Reusable preprocessing pipeline. 

In [2]:
import os
import shutil
import random
import cv2
import pandas as pd

In [3]:
def preprocess_dataset(
        dataset_path,
        output_path="Processed_Dataset",
        image_size=(128,128),
        train_ratio=0.8,
        random_seed=42):

    random.seed(random_seed)

    metadata = []

    # Create output folders
    os.makedirs(output_path, exist_ok=True)

    train_root = os.path.join(output_path, "train")
    val_root = os.path.join(output_path, "validation")

    os.makedirs(train_root, exist_ok=True)
    os.makedirs(val_root, exist_ok=True)

    # Automatically detect classes
    classes = sorted(
        folder for folder in os.listdir(dataset_path)
        if os.path.isdir(os.path.join(dataset_path, folder))
    )

    print("Classes Found:", classes)

    for cls in classes:

        source_folder = os.path.join(dataset_path, cls)

        train_folder = os.path.join(train_root, cls)
        val_folder = os.path.join(val_root, cls)

        os.makedirs(train_folder, exist_ok=True)
        os.makedirs(val_folder, exist_ok=True)

        images = [
            img for img in os.listdir(source_folder)
            if img.lower().endswith(
                (".jpg",".jpeg",".png",".bmp",".gif")
            )
        ]

        random.shuffle(images)

        split = int(len(images) * train_ratio)

        train_images = images[:split]
        val_images = images[split:]

        # Process training images
        for image_name in train_images:

            src = os.path.join(source_folder, image_name)
            dst = os.path.join(train_folder, image_name)

            image = cv2.imread(src)

            if image is None:
                continue

            h, w = image.shape[:2]

            image = cv2.resize(image, image_size)

            cv2.imwrite(dst, image)

            metadata.append({
                "filename": image_name,
                "class": cls,
                "split": "train",
                "original_width": w,
                "original_height": h,
                "resized_width": image_size[0],
                "resized_height": image_size[1],
                "path": dst
            })

        # Process validation images
        for image_name in val_images:

            src = os.path.join(source_folder, image_name)
            dst = os.path.join(val_folder, image_name)

            image = cv2.imread(src)

            if image is None:
                continue

            h, w = image.shape[:2]

            image = cv2.resize(image, image_size)

            cv2.imwrite(dst, image)

            metadata.append({
                "filename": image_name,
                "class": cls,
                "split": "validation",
                "original_width": w,
                "original_height": h,
                "resized_width": image_size[0],
                "resized_height": image_size[1],
                "path": dst
            })

    metadata = pd.DataFrame(metadata)

    metadata.to_csv(
        os.path.join(output_path, "metadata.csv"),
        index=False
    )

    print("\nDataset preprocessing completed.")
    print("Metadata saved.")

    return metadata

In [4]:
metadata = preprocess_dataset(
    dataset_path="cats_dogs_dataset",
    output_path="cats_dogs_preprocessed",
    image_size=(128,128)
)

Classes Found: ['cat', 'dog']

Dataset preprocessing completed.
Metadata saved.


In [6]:
metadata = preprocess_dataset(
    dataset_path="flowers_dataset",
    output_path="flowers_preprocessed",
    image_size=(128,128)
)

Classes Found: ['flowers']

Dataset preprocessing completed.
Metadata saved.
